### 01 - Instalação (Bibliotecas)

In [ ]:
%pip install numpy
%pip install matplotlib
%pip install torch

### 02 - Importação (Recursos)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import time

print(f"Numpy Version: {np.__version__}")
print(f"\nTorch Version: {torch.__version__}")

### 03 - Classe Auxiliar

In [ ]:
class MLP_Classifier:
  def __init__(
      self,
      hidden_layers = [64, 32],
      activation = "relu",
      learning_rate = 0.01,
      epochs = 100,
      patience = 5,
      batch_size = 32,
      optimizer = "adam",
      regularization = None,
      dropout_p = 0.1,
      lambda_l2 = 0.001,
      random_state = None
  ):
    self.hidden_layers = hidden_layers
    self.activation = activation
    self.learning_rate = learning_rate
    self.epochs = epochs
    self.patience = patience
    self.batch_size = batch_size
    self.optimizer = optimizer
    self.regularization = regularization
    self.dropout_p = dropout_p
    self.lambda_l2 = lambda_l2
    self.random_state = random_state

    self.model = None
    self.loss_history = { "train": [], "val": [] }
    self.accuracy_history = { "train": [], "val": [] }
    self.classes_ = None
    self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"Dispositivo: {self.device}")

    print(f"\nCuda: {'Habilitado' if (self.device.type == 'cuda') else 'Desabilitado'}")

    torch.cuda.empty_cache()

    if random_state is not None:
      torch.manual_seed(random_state)

      np.random.seed(random_state)

  def _build_model(self, input_dim, output_dim):
    layers = []

    predict_dim = input_dim

    for dim in self.hidden_layers:
      layers.append(nn.Linear(predict_dim, dim))

      if self.activation == "relu":
        layers.append(nn.ReLU())
      elif self.activation == "tanh":
        layers.append(nn.Tanh())

      if self.regularization == "dropout":
        layers.append(nn.Dropout(p = self.dropout_p))

      predict_dim = dim

    layers.append(nn.Linear(predict_dim, output_dim))

    return nn.Sequential(*layers).to(self.device)
  
  def fit(self, X_train, y_train, X_val = None, y_val = None):
    start = time.time()

    X_train_tensor = torch.FloatTensor(X_train).to(self.device)

    y_train_tensor = torch.LongTensor(y_train).to(self.device)

    if X_val is not None and y_val is not None:
      X_val_tensor = torch.FloatTensor(X_val).to(self.device)

      y_val_tensor = torch.LongTensor(y_val).to(self.device)

      validation_data = (X_val_tensor, y_val_tensor)
    else:
      validation_data = None

    input_dim = X_train.shape[0]

    self.classes_ = torch.unique(y_train_tensor)

    output_dim = len(self.classes_)

    self.model = self._build_model(input_dim, output_dim)

    id = torch.cuda.current_device()

    print(f"GPU em uso: {torch.cuda.get_device_name(id)}")

    criterion = nn.CrossEntropyLoss() if output_dim > 1 else nn.BCEWithLogitsLoss()

    if self.optimizer == "sgd":
      optimizer = optim.SGD(self.model.parameters(), lr = self.learning_rate)
    elif self.optimizer == "adam":
      if self.regularization == "l2":
        optimizer = optim.Adam(self.model.parameters(), lr = self.learning_rate, weight_decay = self.lambda_l2)
      else:
        optimizer = optim.Adam(self.model.parameters(), lr = self.learning_rate, )
    else:
      raise ValueError("O optimizador deve ser 'Adam' ou 'SGD'.")
    
    best_loss = np.inf

    patience_counter = 0

    best_weights = None

    for epoch in range(self.epochs):
      self.model.train().to(self.device)

      epoch_loss = 0.0

      correct = 0

      total = 0

      for i in range(0, len(X_train), self,self.batch_size):
        batch_X = X_train_tensor[i: i + self.batch_size]

        batch_y = y_train_tensor[i: i + self.batch_size]

        optimizer.zero_grad()

        outputs = self.model(batch_X)

        loss = criterion(outputs, batch_y)

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)

        correct += (predicted == batch_y).sum().item()

        total += batch_y.size(0)

      train_loss = epoch_loss / (len(X_train) / self.batch_size)

      train_accuracy = correct / total

      self.loss_history["train"].append(train_loss)

      self.accuracy_history["train"].append(train_accuracy)

      val_loss, val_accuracy = None, None

      if validation_data is not None:
        self.model.eval()

        with torch.no_grad():
          X_val_tensor, y_val_tensor = validation_data

          outputs = self.model(X_val_tensor)

          val_loss = criterion(outputs, y_val_tensor).item()

          _, predicted = torch.max(outputs.data, 1)

          val_accuracy = (predicted == y_val_tensor).sum().item() / len(y_val_tensor)

        self.loss_history["val"].append(val_loss)

        self.accuracy_history["val"].append(val_accuracy)

        if (val_loss < best_loss):
          best_loss = val_loss

          patience_counter = 0

          best_weights = self.model.state_dict()
        else:
          patience_counter += 1

          if (patience_counter >= self.patience):
            print(f"Parada Prematura (Early Stopping)! Época: {epoch + 1}.")

            self.model.load_state_dict(best_weights)

            break

      if ((epoch + 1) % 10 == 0 or epoch == 0):
        print(f"{epoch} | {self.epochs}")
        print(f"Perda (Treinamento): {train_loss:.4f}")
        print(f"Acurácia (Treinamento): {(train_accuracy * 100):.2f}%")

        if (val_loss is not None):
          print(f"Validação (Perda): {val_loss:.4f}")
          print(f"Validação (Acurácia): {(val_accuracy * 100):.2f}%")

    end = time.time()

    training_interval = end - start

    print(f"Duração Total do Treinamento (Segundos): {training_interval:.2f}")

  def predict(self, X):
    if self.model is None:
      raise RuntimeError("O modelo não foi treinado! Execute a função 'Fit'.")
    
    self.model.eval()

    with torch.no_grad():
      X_tensor = torch.FloatTensor(X).to(self.device)

      outputs = self.model(X_tensor)

      _, predictions = torch.max(outputs.data, 1)

    return predictions.cpu().numpy()
  
  def evaluate(self, X, y):
    predictions = self.predict(X)

    accuracy = np.mean(predictions == y)

    return accuracy
  
  def plot_training_history(self):
    fig, (axis_1, axis_2) = plt.subplots(1, 2, figsize=(12, 5))

    axis_1.plot(self.loss_history["train"], label = "Perda (Treinamento)")

    if (self.loss_history["val"]):
      axis_1.plot(self.loss_history["val"], label = "Validação (Perda)")

    axis_1.set_title("Perda e Validação do Treinamento")

    axis_1.set_xlabel("Época")

    axis_1.set_ylabel("Perda")

    axis_1.legend()

    axis_2.plot(self.accuracy_history["train"], label = "Acurácia (Treinamento)")

    if (self.accuracy_history["val"]):
      axis_2.plot(self.accuracy_history["val"], label = "Validação (Acurácia)")

    axis_2.set_title("Acurácia e Validação do Treinamento")

    axis_2.set_xlabel("Época")

    axis_2.set_ylabel("Acurácia")

    axis_2.legend()

    plt.tight_layout()

    plt.show()

  def plot_decision_boundary(self, X, y, step = 0.02):
    if (X.shape[1] != 2):
      print(f"A fronteira de decisão só pode ser renderizada para dados 2D.")

      return
    
    x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1

    y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1

    xx, yy = np.meshgrid(np.arange(x_min, x_max, step), np.arange(y_min, y_max, step))

    grid_points = np.c_[xx.ravel(), yy.ravel()]

    predictions = self.predict(grid_points).reshape(xx.shape)

    plt.contourf(xx, yy, predictions, alpha=0.75)

    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors="k", marker="o")

    plt.title("Fronteira de Decisão")

    plt.show()

### 04 - Aplicação

In [ ]:
# Teste.

test_dataset = np.genfromtxt("./Data/test_dataset.csv", delimiter=",", skip_header=1)

X_test = test_dataset[:, :-1] # Features.

y_test = test_dataset[:, -1] # Labels.

y_test = (y_test + 1) // 2

# Treinamento.

train_dataset = np.genfromtxt("./Data/train_dataset.csv", delimiter=",", skip_header=1)

X_train = train_dataset[:, :-1] # Features.

y_train = train_dataset[:, -1] # Labels.

y_train = (y_train + 1) // 2

# Aplicação.

mlp = MLP_Classifier(
    optimizer="adam",
    learning_rate=0.001,
    hidden_layers=[50, 128, 64, 4],
    activation="relu",
    regularization="dropout",
    dropout_p=0.1
)

mlp.fit(X_train, y_train)

test_accuracy = mlp.evaluate(X_test, y_test)

print(f"Acurácia do Teste: {(test_accuracy * 100):.2f}%")

mlp.plot_training_history()

mlp.plot_decision_boundary(X_train, y_train)

mlp.predict(X_test)

mlp.plot_decision_boundary(X_test, y_test)